# 01 · Data Profiling

**PRT661 Assessment 2 · Group 4 · DKASC Alice Springs**

Purpose: establish the ground truth about the raw data before any ingestion code is written.

Outputs:
* row counts and file sizes per year
* the union of column names across 2008 to 2026
* an explicit schema drift report (which columns exist in which years)
* sample rows from the earliest year, the latest year, and the Master Meter reference file

This notebook writes `Outputs/schema_matrix.csv`, which is cited as evidence in the
Assessment 2 report.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

REPO = Path("/Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting")
RAW  = REPO / "Datasets" / "raw"
REF  = REPO / "Datasets" / "reference"
OUT  = REPO / "Outputs"
OUT.mkdir(exist_ok=True)

files = sorted(RAW.glob("*.csv"))
print(f"{len(files)} raw files, {sum(f.stat().st_size for f in files)/1e9:.2f} GB total")
print(f"pandas {pd.__version__}")

19 raw files, 3.11 GB total
pandas 2.2.3


## 1 · Row counts, column counts and file sizes

This is the slow cell (a couple of minutes). `wc -l` is used instead of reading the files into pandas, which would take far longer.

In [2]:
def count_rows(path):
    out = subprocess.run(["wc", "-l", str(path)], capture_output=True, text=True)
    return int(out.stdout.strip().split()[0]) - 1

profiles = {}
for f in files:
    head = pd.read_csv(f, nrows=3)
    profiles[f.stem] = {
        "columns": list(head.columns),
        "n_cols":  head.shape[1],
        "size_mb": round(f.stat().st_size / 1e6, 1),
        "rows":    count_rows(f),
    }
    p = profiles[f.stem]
    print(f"{f.stem:<24} {p['n_cols']:>3} cols  {p['rows']:>10,} rows  {p['size_mb']:>7} MB")

print(f"\nTOTAL ROWS: {sum(p['rows'] for p in profiles.values()):,}")

Alice_Springs_2008       197 cols      32,046 rows     25.1 MB
Alice_Springs_2009       197 cols     105,404 rows     96.0 MB
Alice_Springs_2010       197 cols     105,408 rows    126.1 MB
Alice_Springs_2011       197 cols     105,191 rows    140.0 MB
Alice_Springs_2012       197 cols     105,684 rows    141.9 MB
Alice_Springs_2013       197 cols     105,365 rows    158.5 MB
Alice_Springs_2014       197 cols     105,399 rows    167.1 MB
Alice_Springs_2015       197 cols     105,408 rows    186.7 MB
Alice_Springs_2016       197 cols     105,696 rows    200.0 MB
Alice_Springs_2017       197 cols     105,366 rows    204.8 MB
Alice_Springs_2018       197 cols     105,408 rows    199.7 MB
Alice_Springs_2019       197 cols     103,193 rows    201.9 MB
Alice_Springs_2020       197 cols     105,670 rows    207.6 MB
Alice_Springs_2021       197 cols     100,833 rows    197.1 MB
Alice_Springs_2022       197 cols     103,191 rows    198.5 MB
Alice_Springs_2023       197 cols     101,902 rows    1

## 2 · Column union and schema matrix

In [3]:
all_cols = []
for p in profiles.values():
    for c in p["columns"]:
        if c not in all_cols:
            all_cols.append(c)

matrix = pd.DataFrame(
    {name: [c in p["columns"] for c in all_cols] for name, p in profiles.items()},
    index=all_cols,
)
matrix.to_csv(OUT / "schema_matrix.csv")
print(f"union of columns across all years: {len(all_cols)}")
print(f"written to {OUT / 'schema_matrix.csv'}")

union of columns across all years: 406
written to /Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting/Outputs/schema_matrix.csv


## 3 · Schema drift report

The key output. Columns present in every year are safe to model on. Columns that appear or disappear partway through define the reconciliation work and the honest limits of the training window.

In [4]:
stable   = matrix.index[matrix.all(axis=1)].tolist()
drifting = matrix.index[~matrix.all(axis=1)].tolist()

print(f"PRESENT IN EVERY YEAR ({len(stable)}):")
for c in stable:
    print("   ", c)

print(f"\nSCHEMA DRIFT ({len(drifting)}):")
for c in drifting:
    yrs = [y for y in matrix.columns if matrix.loc[c, y]]
    print(f"    {c}\n        {len(yrs)} years: {yrs[0]} to {yrs[-1]}")

PRESENT IN EVERY YEAR (14):
    timestamp
    205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received
    205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average
    205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power
    100_DKA_M1_A_Phase_Active_Energy_Delivered_Received
    100_DKA_M1_A_Phase_Current_Phase_Average
    100_DKA_M1_A_Phase_Active_Power
    100_DKA_M1_A_Phase_Performance_Ratio
    103_DKA_M1_B_Phase_Active_Energy_Delivered_Received
    103_DKA_M1_B_Phase_Current_Phase_Average
    103_DKA_M1_B_Phase_Active_Power
    105_DKA_M1_C_Phase_Active_Energy_Delivered_Received
    105_DKA_M1_C_Phase_Current_Phase_Average
    105_DKA_M1_C_Phase_Active_Power

SCHEMA DRIFT (392):
    97_DKA_M10_B_C_Phases_Active_Energy_Delivered_Received
        18 years: Alice_Springs_2008 to Alice_Springs_2025
    97_DKA_M10_B_C_Phases_Current_Phase_Average
        18 years: Alice_Springs_2008 to Alice_Springs_2025
    97_DKA_M10_B_C_Phases_Active_Power
        18 years: Alice_

## 4 · Sample rows

Needed to identify the timestamp column and format, the target column, and which fields are cumulative counters rather than instantaneous readings.

In [5]:
print("=== EARLIEST YEAR ===")
print(pd.read_csv(files[0], nrows=3).to_string(), "\n")

print("=== LATEST YEAR ===")
print(pd.read_csv(files[-1], nrows=3).to_string(), "\n")

print("=== REFERENCE (Master Meter 1) ===")
for f in sorted(REF.glob("*.csv")):
    print(f.name, f"{f.stat().st_size/1e6:.0f} MB")
    print(pd.read_csv(f, nrows=3).to_string())

=== EARLIEST YEAR ===
             timestamp  205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received  205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average  205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power  100_DKA_M1_A_Phase_Active_Energy_Delivered_Received  100_DKA_M1_A_Phase_Current_Phase_Average  100_DKA_M1_A_Phase_Active_Power  100_DKA_M1_A_Phase_Performance_Ratio  103_DKA_M1_B_Phase_Active_Energy_Delivered_Received  103_DKA_M1_B_Phase_Current_Phase_Average  103_DKA_M1_B_Phase_Active_Power  105_DKA_M1_C_Phase_Active_Energy_Delivered_Received  105_DKA_M1_C_Phase_Current_Phase_Average  105_DKA_M1_C_Phase_Active_Power  97_DKA_M10_B_C_Phases_Active_Energy_Delivered_Received  97_DKA_M10_B_C_Phases_Current_Phase_Average  97_DKA_M10_B_C_Phases_Active_Power  78_DKA_M11_3_Phase_Active_Energy_Delivered_Received  78_DKA_M11_3_Phase_Current_Phase_Average  78_DKA_M11_3_Phase_Active_Power  61_DKA_M15_A_Phase_Active_Energy_Delivered_Received  61_DKA_M15_A_Phase_Current